# Оценка: сравнение базовой и дообученной модели

Используем 20 отложенных примеров (`eval_holdout.jsonl`), которые не
участвовали в обучении — только так сравнение отражает реальное обобщение
модели на новые вопросы, а не запоминание train-данных.

Сначала извлекаем чистые instruction и эталонный (целевой) response
из сохранённого prompt-шаблона `<s>[INST]...[/INST]...</s>`.

In [ ]:
import json, re

with open('/content/drive/MyDrive/eval_holdout.jsonl', 'r') as f:
    eval_data = [json.loads(line) for line in f]

def parse_example(example):

    # extract the raw instruction and target response back out of the
    # [INST]...[/INST] template used during training data formatting
    b_open, b_close = chr(91), chr(93)
    pattern = r"<s>" + b_open + r"INST" + b_close + r" (.*?) " + b_open + r"/INST" + b_close + r" (.*?)</s>"
    match = re.match(pattern, example['text'], re.DOTALL)

    instruction_full = match.group(1)
    target_response = match.group(2)
    return {'prompt': instruction_full, 'target': target_response}

parsed_eval = [parse_example(e) for e in eval_data]
print(parsed_eval[0])

{'prompt': 'Approx. how many nurses were enrolled in World War 2?', 'target': 'Over 100,000 nurses were enrolled in the Red Cross during World War II.'}


In [ ]:
!pip install -q transformers peft bitsandbytes accelerate

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
adapter_path = "/content/drive/MyDrive/mistral-laconic-lora-adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# attaches the trained LoRA weights on top of the frozen base model
model = PeftModel.from_pretrained(base_model, adapter_path)

print("Model + adapter loaded")

## Генерация ответов: BASE vs FINE-TUNED

Функция генерации умеет работать в двух режимах через `model.disable_adapter()`
(временно отключает LoRA-влияние без необходимости грузить отдельную копию
модели). `do_sample=False` — жадная (greedy) генерация без случайности,
для честного, воспроизводимого сравнения между версиями модели.

Критично: `model.eval()` вызывается один раз перед первой генерацией.
Без этого модель остаётся в режиме train() после обучения, что активирует
LoRA dropout и приводит к полному распаду генерации в бессвязный текст
(было обнаружено и продиагностировано эмпирически — первая попытка
генерации без eval() дала бессмысленный повторяющийся набор токенов).

In [ ]:
model.eval()  # critical — without this, LoRA dropout stays active during
              # generation (leftover from train() mode), causing incoherent output

def generate_response(prompt, use_adapter=True, max_new_tokens=150):
    input_text = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,        # greedy decoding — deterministic, fair comparison
        repetition_penalty=1.15,  # discourages repeating already-generated tokens
        no_repeat_ngram_size=3,   # blocks repeating the same 3-token sequence
        pad_token_id=tokenizer.eos_token_id,
    )
    if use_adapter:
        with torch.no_grad():
            output = model.generate(**inputs, **gen_kwargs)
    else:
        # temporarily disables LoRA's effect on the forward pass —
        # avoids the memory/time cost of loading a separate base-model copy
        with model.disable_adapter():
            with torch.no_grad():
                output = model.generate(**inputs, **gen_kwargs)
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    response = full_text.split("[/INST]")[-1].strip()
    return response

test_prompt = parsed_eval[0]['prompt']
print("BASE:", generate_response(test_prompt, use_adapter=False))
print("FINE-TUNED:", generate_response(test_prompt, use_adapter=True))
print("TARGET:", parsed_eval[0]['target'])

In [15]:
import json

with open('/content/drive/MyDrive/eval_results.jsonl', 'r') as f:
    results = [json.loads(line) for line in f]

# same example as used in the single-prompt test above (nurses/WWII question) —
# reconstructed from the already-completed full run, no GPU needed
example = results[0]
print("BASE:", example['base_response'])
print("FINE-TUNED:", example['finetuned_response'])
print("TARGET:", example['target'])

BASE: It is estimated that approximately 90,000 to 130,500 nurses served during World War II in various military nursing services from different countries. The majority of these nurses were from the United States, with around 68,047 serving in the U.S. Army and Navy Nurses Corps. Other significant contributions came from the British Empire (around 25,052), Canada (approximately 12,035), Australia (about 3,515), New Zealand (around1,113), and other Allied nations. These numbers are approximate and may vary slightly depending on the source.
FINE-TUNED: About 10,000 nurses served during the war.
TARGET: Over 100,000 nurses were enrolled in the Red Cross during World War II.


In [ ]:
results = []
for i, ex in enumerate(parsed_eval):
    # same generate_response function as the single-example test above,
    # now applied across all 20 held-out examples
    base_resp = generate_response(ex['prompt'], use_adapter=False)
    ft_resp = generate_response(ex['prompt'], use_adapter=True)
    results.append({
        'prompt': ex['prompt'],
        'target': ex['target'],
        'base_response': base_resp,
        'finetuned_response': ft_resp,
    })
    print(f"{i+1}/20 done")

with open('/content/drive/MyDrive/eval_results.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print("Saved")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
with open('/content/drive/MyDrive/eval_results.jsonl', 'r') as f:
    results = [json.loads(line) for line in f]

print(f"Loaded {len(results)} examples")


Mounted at /content/drive
Loaded 20 examples


## Метрика: ROUGE

ROUGE измеряет пересечение слов/подпоследовательностей между сгенерированным
текстом и эталоном. ROUGE-1 — по отдельным словам, ROUGE-L — по самой
длинной общей подпоследовательности (чувствительнее к порядку слов).
`use_stemmer=True` — приводит слова к основе (running - run), чтобы не
штрафовать за грамматические формы.

Дополнительно считаем среднюю длину ответов — прямой количественный
показатель лаконичности, отдельно от смыслового совпадения с эталоном.

In [6]:
!pip install -q rouge-score

from rouge_score import rouge_scorer
import json

with open('/content/drive/MyDrive/eval_results.jsonl', 'r') as f:
    results = [json.loads(line) for line in f]

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

base_scores = {'rouge1': [], 'rougeL': []}
ft_scores = {'rouge1': [], 'rougeL': []}

for r in results:
    base_s = scorer.score(r['target'], r['base_response'])
    ft_s = scorer.score(r['target'], r['finetuned_response'])

    base_scores['rouge1'].append(base_s['rouge1'].fmeasure)
    base_scores['rougeL'].append(base_s['rougeL'].fmeasure)
    ft_scores['rouge1'].append(ft_s['rouge1'].fmeasure)
    ft_scores['rougeL'].append(ft_s['rougeL'].fmeasure)

import numpy as np
print("BASE  — ROUGE-1: {:.3f}, ROUGE-L: {:.3f}".format(
    np.mean(base_scores['rouge1']), np.mean(base_scores['rougeL'])))
print("FT    — ROUGE-1: {:.3f}, ROUGE-L: {:.3f}".format(
    np.mean(ft_scores['rouge1']), np.mean(ft_scores['rougeL'])))


base_lens = [len(r['base_response'].split()) for r in results]
ft_lens = [len(r['finetuned_response'].split()) for r in results]
print(f"\nMean Length of BASE: {np.mean(base_lens):.1f} words")
print(f"Mean Length of FT:   {np.mean(ft_lens):.1f} words")

  Preparing metadata (setup.py) ... done
BASE  — ROUGE-1: 0.251, ROUGE-L: 0.171
FT    — ROUGE-1: 0.338, ROUGE-L: 0.234

Mean Length of BASE: 80.0 words
Mean Length of FT:   59.3 words


In [7]:
for r in results[:5]:
    print("PROMPT:", r['prompt'])
    print("TARGET:", r['target'])
    print("BASE:", r['base_response'])
    print("FT:", r['finetuned_response'])
    print("---")

PROMPT: Approx. how many nurses were enrolled in World War 2?
TARGET: Over 100,000 nurses were enrolled in the Red Cross during World War II.
BASE: It is estimated that approximately 90,000 to 130,500 nurses served during World War II in various military nursing services from different countries. The majority of these nurses were from the United States, with around 68,047 serving in the U.S. Army and Navy Nurses Corps. Other significant contributions came from the British Empire (around 25,052), Canada (approximately 12,035), Australia (about 3,515), New Zealand (around1,113), and other Allied nations. These numbers are approximate and may vary slightly depending on the source.
FT: About 10,000 nurses served during the war.
---
PROMPT: Who won Euro song contest Save All Your Kisses For Me
TARGET: Brotherhood of Man
BASE: "Save All Your kisses for Me" was not a winning entry in the Eurovision Song Contest. The song was performed by the Swedish group Brotherhood of Man at the 1976 contes

## Метрика: LLM-as-a-judge

ROUGE механически считает пересечение слов, но не понимает смысл — не
отличит "убрали лишние оговорки" от "потеряли важный факт", если оба
случая одинаково меняют совпадение слов с эталоном. LLM-судья читает
осмысленно и оценивает по двум независимым осям:

- **Conciseness** — насколько ответ лаконичен без потери сути
- **Factual accuracy** — сохранены ли ключевые факты относительно
  эталонного (target) ответа

Судья: Llama 3.3 70B через Groq API (тот же провайдер, что и для style
transfer в Шаге 1) — `temperature=0` для максимально консистентных,
воспроизводимых оценок.

In [ ]:
!pip install groq
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def judge_response(prompt, target, model_response):
    judge_prompt = f"""You are evaluating an AI assistant's answer for conciseness and factual accuracy.

    Question: {prompt}
    Reference answer (target style - concise expert): {target}
    Model's answer: {model_response}

    Rate the model's answer on two dimensions, each from 1 to 5:
    1. Conciseness: is it appropriately brief and to the point, without unnecessary padding?
    2. Factual accuracy: does it preserve the key facts from the reference answer, without inventing or omitting important information?

    Output ONLY two numbers separated by a comma, e.g. "4,3". No explanation."""

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0,  # deterministic scoring — avoids sampling noise skewing the judge's ratings
    )
    return completion.choices[0].message.content.strip()

test = results[0]
print("BASE judge:", judge_response(test['prompt'], test['target'], test['base_response']))
print("FT judge:", judge_response(test['prompt'], test['target'], test['finetuned_response']))

BASE judge: 2,2
FT judge: 5,1

In [ ]:
import time, re

def parse_scores(text):

    # regex instead of a plain .split(',') — a small safeguard in case
    # the model adds stray whitespace or extra text around the two numbers

    match = re.search(r'(\d)\s*,\s*(\d)', text)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

judge_results = []
for i, r in enumerate(results):
    base_raw = judge_response(r['prompt'], r['target'], r['base_response'])
    time.sleep(2)
    ft_raw = judge_response(r['prompt'], r['target'], r['finetuned_response'])
    time.sleep(2)

    base_conc, base_acc = parse_scores(base_raw)
    ft_conc, ft_acc = parse_scores(ft_raw)

    judge_results.append({
        'prompt': r['prompt'],
        'base_conciseness': base_conc, 'base_accuracy': base_acc,
        'ft_conciseness': ft_conc, 'ft_accuracy': ft_acc,
    })
    print(f"{i+1}/20: BASE(conciseness={base_conc}, accuracy={base_acc})  FT(conciseness={ft_conc}, accuracy={ft_acc})")

import numpy as np
print("\n=== Average scores ===")
print(f"BASE — Conciseness: {np.mean([r['base_conciseness'] for r in judge_results]):.2f}, Accuracy: {np.mean([r['base_accuracy'] for r in judge_results]):.2f}")
print(f"FT   — Conciseness: {np.mean([r['ft_conciseness'] for r in judge_results]):.2f}, Accuracy: {np.mean([r['ft_accuracy'] for r in judge_results]):.2f}")

with open('/content/drive/MyDrive/judge_results.jsonl', 'w') as f:
    for r in judge_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print("\nSaved to judge_results.jsonl")

1/20: BASE(conciseness=2, accuracy=2)  FT(conciseness=5, accuracy=1)
2/20: BASE(conciseness=2, accuracy=1)  FT(conciseness=2, accuracy=1)
3/20: BASE(conciseness=3, accuracy=5)  FT(conciseness=5, accuracy=5)
4/20: BASE(conciseness=2, accuracy=5)  FT(conciseness=2, accuracy=2)
5/20: BASE(conciseness=2, accuracy=3)  FT(conciseness=2, accuracy=3)
6/20: BASE(conciseness=2, accuracy=4)  FT(conciseness=2, accuracy=4)
7/20: BASE(conciseness=2, accuracy=2)  FT(conciseness=3, accuracy=3)
8/20: BASE(conciseness=2, accuracy=2)  FT(conciseness=3, accuracy=5)
9/20: BASE(conciseness=2, accuracy=1)  FT(conciseness=2, accuracy=1)
10/20: BASE(conciseness=2, accuracy=1)  FT(conciseness=5, accuracy=2)
11/20: BASE(conciseness=2, accuracy=5)  FT(conciseness=5, accuracy=5)
12/20: BASE(conciseness=3, accuracy=2)  FT(conciseness=3, accuracy=2)
13/20: BASE(conciseness=2, accuracy=1)  FT(conciseness=1, accuracy=1)
14/20: BASE(conciseness=2, accuracy=1)  FT(conciseness=3, accuracy=1)
15/20: BASE(conciseness=2, ac

## Детальный breakdown по примерам

Средние баллы (выше) могут маскировать неоднородную картину — разбиваем
20 примеров на категории изменений, чтобы понять не только "стало лучше
или хуже в среднем", но и насколько равномерно проявляется эффект style
transfer по разным вопросам.

In [ ]:
import numpy as np

conc_improved = sum(1 for r in judge_results if r['ft_conciseness'] > r['base_conciseness'])
acc_dropped = sum(1 for r in judge_results if r['ft_accuracy'] < r['base_accuracy'])
acc_improved = sum(1 for r in judge_results if r['ft_accuracy'] > r['base_accuracy'])
identical = sum(1 for r in judge_results if r['ft_conciseness'] == r['base_conciseness'] and r['ft_accuracy'] == r['base_accuracy'])

print(f"Examples where FT is more concise than BASE: {conc_improved}/20")
print(f"Examples where FT accuracy dropped: {acc_dropped}/20")
print(f"Examples where FT accuracy improved: {acc_improved}/20")
print(f"Examples with identical scores (no visible change): {identical}/20")

Examples where FT is more concise than BASE: 9/20
Examples where FT accuracy dropped: 6/20
Examples where FT accuracy improved: 4/20
Examples with identical scores (no visible change): 6/20
